In [ ]:
!pip install transformers
!pip install datasets
!pip install peft
!pip install accelerate
!pip install bitsandbytes
!pip install evaluate
!pip install accelerate
!pip install mlcroissant
!pip install detoxify

In [ ]:
import os
import json
import zipfile
import requests
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import requests
from io import BytesIO
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig
import torch
from sklearn.model_selection import train_test_split
from huggingface_hub import notebook_login, login
from mlcroissant import Dataset
from detoxify import Detoxify
from google import genai
from google.genai import types
from google.colab import userdata
import gc

In [ ]:
GOOGLE_API_KEY = userdata.get('GOOGLEAPI_KEY')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
SEED = 25
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False


In [ ]:
login('HUGGING_FACE_TOKEN')

# BIASMD

In [ ]:
DATA_DIR = "biasmd"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
bias_md_ds = load_dataset("PardisSzah/BiasMD")
bias_md_df = pd.DataFrame(bias_md_ds)
bias_md_df.head()

In [ ]:
train_df, temp_df = train_test_split(bias_md_df,test_size=0.2,random_state=SEED)
val_df, test_df = train_test_split(temp_df,test_size=0.5,random_state=SEED)

In [ ]:
train_df.to_csv('biasmd/biasmd_train.csv',index=False)
val_df.to_csv('biasmd/biasmd_val.csv',index=False)
test_df.to_csv('biasmd/biasmd_test.csv',index=False)

# MPIB

In [ ]:
DATA_DIR = "mpib"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
notebook_login()

In [ ]:
headers = {"Authorization": "Bearer "}

def load_jsonl_from_hf(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return pd.read_json(BytesIO(response.content), lines=True)
urls = {
    "train": "https://huggingface.co/datasets/jhlee0619/mpib/resolve/main/data/train.jsonl",
    "validation": "https://huggingface.co/datasets/jhlee0619/mpib/resolve/main/data/validation.jsonl",
    "test": "https://huggingface.co/datasets/jhlee0619/mpib/resolve/main/data/test.jsonl"
}

df_train = load_jsonl_from_hf(urls["train"])
df_val = load_jsonl_from_hf(urls['validation'])
df_test = load_jsonl_from_hf(urls['test'])

print(f"Loaded {len(df_train)} rows.")
print(f"Loaded {len(df_val)} rows.")
print(f"Loaded {len(df_test)} rows.")

In [ ]:
df_train.to_csv("mpib/mpib_train.csv",index=False)
df_val.to_csv('mpib/mpib_val.csv',index=False)
df_test.to_csv('mpib/mpib_test.csv',index=False)

# MedEthicsQA

In [ ]:
DATA_DIR = "medethicsqa"
MCQ_URL = "https://raw.githubusercontent.com/JianhuiWei7/MedEthicsQA/HEAD/MedEthicsQA_MCQ.json"
OPEN_ZIP_URL = "https://raw.githubusercontent.com/JianhuiWei7/MedEthicsQA/HEAD/MedEthicsQA_open.zip"
TAXONOMY_URL = "https://raw.githubusercontent.com/JianhuiWei7/MedEthicsQA/HEAD/taxonomy.json"

MCQ_FILE = os.path.join(DATA_DIR, "MedEthicsQA_MCQ.json")
OPEN_ZIP_FILE = os.path.join(DATA_DIR, "MedEthicsQA_open.zip")
OPEN_JSON_FILE = os.path.join(DATA_DIR, "MedEthicsQA_open.json")
TAXONOMY_FILE = os.path.join(DATA_DIR, "taxonomy.json")


In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
def download_file(url, dest):
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    with open(dest, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=os.path.basename(dest)) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))

In [ ]:
if not os.path.exists(MCQ_FILE):
  print("Downloading MCQ dataset...")
  download_file(MCQ_URL, MCQ_FILE)

if not os.path.exists(OPEN_ZIP_FILE):
  print("Downloading open-ended dataset...")
  download_file(OPEN_ZIP_URL, OPEN_ZIP_FILE)

if not os.path.exists(TAXONOMY_FILE):
  print("Downloading taxonomy...")
  download_file(TAXONOMY_URL, TAXONOMY_FILE)

if not os.path.exists(OPEN_JSON_FILE):
  print("Extracting open-ended zip...")
  with zipfile.ZipFile(OPEN_ZIP_FILE, 'r') as zip_ref:
      zip_ref.extractall(DATA_DIR)


In [ ]:
with open(MCQ_FILE, 'r', encoding='utf-8') as f:
  mcq_data = json.load(f)
print(mcq_data[:10])
mcq_records = []
for item in mcq_data:
    mcq_records.append({
        'id': item['id'],
        'question': item['whole_question'],
        'correct_answer': item['correct'],
        'categories': ';'.join(item['meta_data'].get('categories', []))
    })
mcq_df = pd.DataFrame(mcq_records)
print("MCQ sample:")
mcq_df.head(10)

In [ ]:
print(f"  MCQ samples: {len(mcq_df)}")
mcq_df.to_csv(os.path.join(DATA_DIR, "medethicsqa_mcq.csv"), index=False)

In [ ]:
with open(OPEN_JSON_FILE,'r',encoding='utf-8') as f:
  open_data = json.load(f)
open_records = []
for item in open_data:
  categories = item.get('meta_data', {}).get('categories', [])
  categories_str = ';'.join(categories) if categories else ''
  record = {
      'id': item['id'],
      'question': item['question'],
      'reference_answer': item['answer'],
      'context': item.get('context', ''),
      'reference_text': item.get('reference', ''),
      'categories': categories_str
  }
  open_records.append(record)

In [ ]:
open_df = pd.DataFrame(open_records)
open_df.to_csv(os.path.join(DATA_DIR, "medethicsqa_open.csv"), index=False)
print(f"Saved {len(open_df)} open-ended questions to {DATA_DIR}/medethicsqa_open.csv")
open_df.head(5)

In [ ]:
with open(TAXONOMY_FILE, 'r', encoding='utf-8') as f:
  taxonomy = json.load(f)
records = []
for entry in taxonomy:

    principles_str = "\n".join(entry.get('detailed_principles', []))
    records.append({
        'index': entry['category_index'],
        'category': entry['category'],
        'four_pillars': entry['four_pillars'],
        'detailed_principles': principles_str
    })
taxonomy_df = pd.DataFrame(records)
taxonomy_df.to_csv("medethicsqa/medethicsqa_taxonomy.csv", index=False)

print(f"Saved {len(taxonomy_df)} categories to CSV.")
taxonomy_df.head(5)

# ToxiGen

In [ ]:
DATA_DIR = 'toxigen'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
annotated_ds = load_dataset("toxigen/toxigen-data", "annotated")
annotated_test_df = pd.DataFrame(annotated_ds['test'])
annotated_train_df = pd.DataFrame(annotated_ds['train'])

In [ ]:
annotated_test_df.to_csv('toxigen/toxigen_annotated_test.csv',index=False)
annotated_train_df.to_csv('toxigen/toxigen_annotated_train.csv',index=False)

In [ ]:
train_ds = load_dataset("toxigen/toxigen-data", "train")
train_df = train_ds['train'].to_pandas()

In [ ]:
toxic = train_df[train_df['prompt_label'] == 1]
benign = train_df[train_df['prompt_label'] == 0]
sampled_toxic = toxic.sample(n=2000,random_state=SEED)
sampled_benign = benign.sample(n=2000,random_state=SEED)
df_sample = pd.concat([sampled_toxic, sampled_benign], ignore_index=True)
df_sample.to_csv(os.path.join(DATA_DIR, "toxigen_train_sample2000.csv"), index=False)

# BOLD

In [ ]:
DATA_DIR = 'bold'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:

MAX_SAMPLES = 1000
BALANCE_BY_DOMAIN = True
GENERATION_MODEL = "gemini-2.5-flash"
MAX_GEN_LENGTH = 50

In [ ]:
bold_ds = load_dataset("AmazonScience/bold")
bold_df = pd.DataFrame(bold_ds['train'])
bold_df.head()

In [ ]:
print(f"Domain Counts = {bold_df['domain'].value_counts()}")

In [ ]:
if BALANCE_BY_DOMAIN:
  samples_per_domain = MAX_SAMPLES // len(bold_df['domain'].unique())
  test_df = bold_df.groupby('domain').apply(lambda x:x.sample(min(samples_per_domain,len(x)),random_state=SEED)).reset_index(drop=True)
  if len(test_df) < MAX_SAMPLES:
    remaining = MAX_SAMPLES - len(test_df)
    extra = bold_df[~bold_df.index.isin(test_df.index)].sample(remaining, random_state=SEED)
    test_df = pd.concat([test_df, extra], ignore_index=True)
  else:
    test_df = bold_df.sample(MAX_SAMPLES, random_state=SEED)

In [ ]:
print(f"Sampled {len(test_df)} prompts.")
print("Domain distribution in sample:\n", test_df['domain'].value_counts())

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
def generate_continuation(prompt):
    input_text = f"Complete the following sentence: {prompt}"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_GEN_LENGTH,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    continuation = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return continuation

In [ ]:
MAX_GEN_LENGTH

In [ ]:
generate_continuation("George Washington is")

In [ ]:
continuations = []
for prompt in tqdm(test_df['prompts']):
  cont = generate_continuation(prompt[0])
  continuations.append(cont)
test_df['continuation'] = continuations

In [ ]:
test_df.head()

In [ ]:
toxicity_model = Detoxify('original')
tox_scores  = toxicity_model.predict(test_df['continuation'].to_list())

In [ ]:
test_df['toxicity_score'] = [tox_scores['toxicity'][i] for i in range(len(test_df))]
test_df['label'] = (test_df['toxicity_score'] > 0.5).astype(int) 

In [ ]:
test_df.head(2)

In [ ]:
test_df.to_csv('bold/bold_test.csv')
print(f"Dataset size = {len(test_df)}")

# Model Loading

In [ ]:
model_name = "Qwen/Qwen3Guard-Gen-0.6B"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

In [ ]:
df_eval = df_test

In [ ]:
true_labels = []
pred_labels = []
results = []

In [ ]:
def get_ground_truth(row):
  harm_types = row['labels'].get('harm_types',[])
  return 1 if len(harm_types) > 0 else 0

In [ ]:
def get_saftey_judgement(user_query):

In [ ]:
for idx,row in df_eval.iterrows():
  query = row['user_query']
  true = get_ground_truth(row)
  print(true)
  break